In [1]:
%reset -f
import arcpy
arcpy.ResetEnvironments()
import os
import csv

In [2]:
# ── CS METHOD: FEATURE CLASS PATHS ───────────────────────────────────────────────────────────────────

# Custom lists of feature classes whose properties will be written to each output CSV - each list outputs one CSV with each feature class in list as a row in CSV and each property as a column; outputs CSVs named by list name (i.e., "merged_sightings_and_inputs" list -> merged_sightings_and_inputs.csv)
csv_groups = {
    "reproject_Caribbeanshorelinesallg4326_vs_Caribbeanshorelinesallp26920": [
        r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Shoreline_geodatabases\Caribbean_shorelines.gdb\Caribbean_shorelines_all_g4326",
        r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Shoreline_geodatabases\Caribbean_shorelines.gdb\Caribbean_shorelines_all_p26920"
    ],
    "clip2boundary_Caribbeanshorelinesall_vs_CARIBBEANSHORELINESTHESIS_p26920": [
        r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Shoreline_geodatabases\Caribbean_shorelines.gdb\Caribbean_shorelines_all_p26920",
        r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Shoreline_geodatabases\Caribbean_shorelines.gdb\CARIBBEAN_SHORELINES_THESIS_p26920"
    ],
    "compareUTMzones_boundarythesis_19N_vs_20N_p26920": [
        r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Thesis_geodatabases\misc_thesis\thesis_boundary_zone20N_zone19N\boundary_zone_comparison.gdb\boundary_thesis_UTM_zone19N_p26920",
        r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Thesis_geodatabases\misc_thesis\thesis_boundary_zone20N_zone19N\boundary_zone_comparison.gdb\boundary_thesis_UTM_zone20N_p26920"
    ],
    "compareUTMzones_mgrs19PQ_p3857_vs_p26919_vs_p26920": [
        r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Thesis_geodatabases\misc_thesis\thesis_boundary_zone20N_zone19N\boundary_zone_comparison.gdb\esri_19_combinePQ_mgrsUTM_grid_p3857",
        r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Thesis_geodatabases\misc_thesis\thesis_boundary_zone20N_zone19N\boundary_zone_comparison.gdb\esri_19_combinePQ_mgrsUTM_grid_p26919",
        r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Thesis_geodatabases\misc_thesis\thesis_boundary_zone20N_zone19N\boundary_zone_comparison.gdb\esri_19_combinePQ_mgrsUTM_grid_p26920"
    ],
    "properties_esriglobalmgrsUTMgridp3857":[
        r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Thesis_geodatabases\misc_thesis\thesis_boundary_zone20N_zone19N\boundary_zone_comparison.gdb\esri_global_mgrsUTM_grid_p3857"
    ],
    "rdomainvsoutlinevsfootprint_rdomainUSC10mX_vs_outlineUSC10mX_vs_footprintUSC10mX_p26920v6642" : [
        r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\Jeremiah_NOAA_10meter_USCaribbean.gdb\rdomain_USC10m_bathy_p26920_v6642",
        r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\Jeremiah_NOAA_10meter_USCaribbean.gdb\outline_USC10m_bathy_p26920_v6642",
        r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\MASTER_RASTERS_USC_GBC_MERGED.gdb\footprint_USC10m_bathy_p26920_v6642",
        r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\Jeremiah_NOAA_10meter_USCaribbean.gdb\rdomain_USC10m_slope_p26920_v6642",
        r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\Jeremiah_NOAA_10meter_USCaribbean.gdb\outline_USC10m_slope_p26920_v6642",
        r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\MASTER_RASTERS_USC_GBC_MERGED.gdb\footprint_USC10m_slope_p26920_v6642"
    ]
}

# Folder to save output CSVs to (each CSV will be named based on the list it contains info for)
csv_folder = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\metadata_and_information_csvs\featureclass_information_csvs"

In [3]:
# ── SETTINGS ─────────────────────────────────────────────────────────────────────────────────────────
arcpy.env.overwriteOutput = True    # overwrite existing outputs on rerun

In [4]:
# ── DICTIONARY OF FEATURE CLASS PROPERTIES TO INCLUDE IN CSV ─────────────────────────────────────────

def get_feature_class_properties(fc_path):
    # get the Describe object (contains name, shape type, spatial reference, extent, etc.)
    desc = arcpy.Describe(fc_path)
    # get feature count (number of rows/features in the feature class)
    feature_count = int(arcpy.management.GetCount(fc_path)[0])
    # build dictionary mapping each property label to its value
    return {
        # store input path so you can trace each feature class back to source
        "Path": fc_path,
        # dataset name without full path
        "Name": desc.name,
        # feature class type (e.g. Simple)
        "FeatureType": desc.featureType,
        # geometry type (e.g. Point, Polyline, Polygon)
        "ShapeType": desc.shapeType,
        # number of features
        "FeatureCount": feature_count,
        # number of fields in the attribute table
        "FieldCount": len(desc.fields),
        # comma-separated list of field names
        "Fields": ", ".join(f.name for f in desc.fields),
        # name of the geometry field (usually Shape)
        "ShapeFieldName": desc.shapeFieldName,
        # name of the Object ID field (usually OBJECTID)
        "OIDFieldName": desc.OIDFieldName,
        # whether geometry has Z (elevation) values
        "HasZ": desc.hasZ,
        # whether geometry has M (measure) values
        "HasM": desc.hasM,
        # name of coordinate system (projected name if projected, else geographic)
        "SpatialRef": desc.spatialReference.name,
        # geographic coordinate system name (underlying datum/GCS)
        "GeogCS": desc.spatialReference.GCS.name if desc.spatialReference.GCS else None,
        # vertical coordinate system name (returns "None" string if feature class has no vertical CS defined)
        "VertCS": desc.spatialReference.VCS.name if desc.spatialReference.VCS and desc.spatialReference.VCS.name else None,
        # spatial extent (xmin, ymin, xmax, ymax) columns
        "Extent_Xmin": desc.extent.XMin,
        "Extent_Ymin": desc.extent.YMin,
        "Extent_Xmax": desc.extent.XMax,
        "Extent_Ymax": desc.extent.YMax,
    }

In [5]:
# ── COMPILE FEATURE CLASS PROPERTIES INTO CSV DATA ───────────────────────────────────────────────────

# Function to write feature class properties to CSV (1 feature class per row, 1 property per column)
def export_feature_class_properties(feature_classes, out_csv):
    # if single path string was passed, wrap it in list so loop works
    if isinstance(feature_classes, str):
        feature_classes = [feature_classes]

    # list that will hold one dictionary (row) per feature class
    rows = []
    # loop over every feature class path provided
    for fc in feature_classes:
        try:
            # collect properties of feature class and add them as a single row
            rows.append(get_feature_class_properties(fc))
        # print warning if feature class fails and continue to next feature class in group
        except Exception as e:
            print(f"❌[FAILURE] CSV data failed to write for feature class {os.path.basename(fc)}: {e} \n")

    # if nothing was collected, warn and stop before writing empty file
    if not rows:
        print(f"⚠️[WARNING] No properties collected (empty CSV file): {os.path.basename(out_csv)} \n")
        return

    # use keys from the first row as CSV column headers
    fieldnames = list(rows[0].keys())
    # open output CSV for writing (newline="" avoids blank lines on Windows)
    with open(out_csv, "w", newline="") as f:
        # create writer that maps dictionaries to CSV columns
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        # write header row
        writer.writeheader()
        # write one row per feature class dictionary
        writer.writerows(rows)

    # print message confirming how many feature classes were written and to where
    print(f"✅ Wrote {len(rows)} feature class(es) to CSV: {out_csv} \n")

In [6]:
# ── CS METHOD: CALL FUNCTION AND EXPORT CSVs (1 CSV PER LIST) ────────────────────────────────────────

# loop over each named group and its list of feature classes
for csv_name, feature_classes in csv_groups.items():
    # build the full output path from the group name (e.g. sightings_selection.csv)
    out_csv = os.path.join(csv_folder, f"{csv_name}.csv")
    # call function to write all feature classes in this group into their own combined CSV
    export_feature_class_properties(feature_classes, out_csv)

✅ Wrote 2 feature class(es) to CSV: C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\metadata_and_information_csvs\featureclass_information_csvs\reproject_Caribbeanshorelinesallg4326_vs_Caribbeanshorelinesallp26920.csv 

✅ Wrote 2 feature class(es) to CSV: C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\metadata_and_information_csvs\featureclass_information_csvs\clip2boundary_Caribbeanshorelinesall_vs_CARIBBEANSHORELINESTHESIS_p26920.csv 

✅ Wrote 2 feature class(es) to CSV: C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\metadata_and_information_csvs\featureclass_information_csvs\compareUTMzones_boundarythesis_19N_vs_20N_p26920.csv 

✅ Wrote 3 feature class(es) to CSV: C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\metadata_and_information_csvs\featureclass_information_csvs\compareUTMzones_mgrs19PQ_p3857_vs_p26919_vs_p26920.csv 

✅ Wrote 1 feature class(es) to CSV: C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\metadata_and_information_csvs\featureclass_information_csv